# LLM-AggreFact: windowed NLI evaluation (ENTAILMENT-ONLY, no GED)

Fast GPU re-run of the `mcp-hallucination-checker` entailment evaluation on a Kaggle T4.

**What this measures.** Only the DeBERTa NLI component (`cross-encoder/nli-deberta-v3-base`). The GED / structural
graph comparison of the full pipeline is **not** part of this notebook, so these numbers must not be reported as
numbers for the whole system.

**What changed vs. the old baseline.** The old run fed the whole document to the model and let the tokenizer truncate
at 512 tokens, discarding evidence that appears late in long documents. Here each document is split into
180-word windows with 40-word overlap, every (window, claim) pair is scored, and the **max entailment probability**
over windows is the claim score (identical to `server/pipeline/entailment.py`).

**Protocol.** Threshold is calibrated on the DEV split (maximise balanced accuracy), then the TEST split is evaluated
once at that threshold. The repo's fixed threshold of 0.40 is also reported for comparison, but it was not tuned on this data.

### How to get the data onto Kaggle
1. Kaggle > Datasets > *New Dataset*, upload `dev-00000-of-00001.parquet` and `test-00000-of-00001.parquet`
   (from the repo root), name it e.g. `llm-aggrefact`.
2. In this notebook: *Add Input* > your dataset. Files appear under `/kaggle/input/llm-aggrefact/`.
3. Settings: Accelerator = **GPU T4**, Internet = **On** (needed once to download the model from Hugging Face;
   or attach the model as a Kaggle Model/Dataset and set `MODEL_PATH` below).

**Added: MiniCheck baseline (section 8c).** Optional three-way comparison (truncation DeBERTa / windowed DeBERTa / MiniCheck) on the same TEST rows with DEV-tuned thresholds, bootstrap CIs and a paired bootstrap. Needs Internet = On for MiniCheck; set `RUN_MINICHECK = False` to skip it.


## 1. Setup

In [ ]:
import subprocess, sys, importlib

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Kaggle images already ship torch/transformers/pandas; only the tokenizer deps may be missing.
_ensure("sentencepiece")
_ensure("protobuf", "google.protobuf")
_ensure("tqdm")

import os, glob, json, time, math, hashlib, re
import numpy as np, pandas as pd, torch, transformers
from tqdm.auto import tqdm

print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU. Enable Settings > Accelerator > GPU T4. The run will be very slow on CPU.")

# ---- Data location ---------------------------------------------------------
DATA_DIR = "/kaggle/input/llm-aggrefact"      # expected dataset slug; falls back to a recursive search

def find_parquet(split):
    pats = [f"{DATA_DIR}/{split}-*.parquet", f"{DATA_DIR}/**/{split}-*.parquet",
            f"/kaggle/input/**/{split}-*.parquet", f"./{split}-*.parquet", f"../**/{split}-*.parquet"]
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits
    raise FileNotFoundError(
        f"No {split}-*.parquet found. Upload the LLM-AggreFact parquet shards as a Kaggle dataset "
        f"and attach it to this notebook (see the instructions at the top).")

PATHS = {s: find_parquet(s) for s in ("dev", "test")}
print(json.dumps(PATHS, indent=2))

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./kaggle_working"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Configuration

In [ ]:
# ---------------- Config ----------------
MODEL_NAME = "cross-encoder/nli-deberta-v3-base"   # same as server/pipeline/entailment.py
MODEL_PATH = None            # None = auto-detect an attached model dataset under /kaggle/input (folder with config.json + model.safetensors),
                             # else download MODEL_NAME from Hugging Face (needs Internet = On)

WINDOW  = 180                # words per premise window
OVERLAP = 40                 # word overlap between consecutive windows (stride = WINDOW - OVERLAP = 140)
MAX_LEN = 512                # tokenizer max length (final truncation safety guard, as in the repo)

HYPOTHESIS_MODE = "raw"      # "raw" : feed the claim unchanged (standard NLI usage; recommended)
                             # "repo": reproduce the repo's _make_hypothesis() templating (what the old baseline used)
                             # "raw" : feed the claim unchanged (standard NLI usage; useful as an ablation)

# Sampling: None = full split. An int draws a seeded sample stratified by (source dataset, label).
DEV_LIMIT  = None            # e.g. 3000 for a quick calibration run
TEST_LIMIT = None            # e.g. 2000 for a smoke run; use None for the real evaluation
SEED = 42

FIXED_THRESHOLD = 0.40       # configs/thresholds.yaml entailment_threshold
N_BOOT = 1000                # bootstrap resamples for 95% CIs

# Old truncation baseline (full 29,320-row test run recorded in STATUS.md)
OLD_BASELINE = {"accuracy": 0.4228, "balanced_accuracy": 0.5848, "precision": 0.8925, "recall": 0.2942,
                "f1": 0.4425, "tp": 6717, "fp": 809, "tn": 5679, "fn": 16115, "threshold": 0.40}
RUN_TEMPLATE_ABLATION = True   # also score a 2k test sample with the repo's templated hypotheses (old-baseline behaviour)
RUN_TRUNC_BASELINE = True    # also re-score a 2k test sample with plain 512-token truncation (apples-to-apples)   # True: also re-score a 2k test sample with plain 512-token truncation (apples-to-apples)
TRUNC_SAMPLE = 2000

TOKEN_BUDGET = 24576         # padded tokens per batch (batch = budget // longest_len). Lower if you hit OOM; auto-halves on OOM.
MAX_BATCH = 256
CKPT_EVERY_SEC = 120         # checkpoint partial scores this often

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Scoring core
Reimplementation of the repo's windowing and label handling:
`_premise_windows` (180/40 words, tail windows kept), `_make_hypothesis` (templating, copied verbatim),
softmax over the 3 NLI logits, entailment index read from `config.id2label`, and max over windows per claim.
All (record, window) pairs are flattened, sorted by token length, batched under a token budget in fp16, and the
partial scores are checkpointed so an interrupted run can resume.

In [ ]:
# ---- Repo-faithful text handling -------------------------------------------------------------
def make_hypothesis(span: str) -> str:
    # Verbatim port of server.pipeline.entailment._make_hypothesis
    stripped = span.strip()
    if re.match(r"^[\$â‚¬Â£Â¥]?[\d,]+(\.\d+)?(%|[kKmMbB])?$", stripped) or \
       re.match(r"^v?\d+\.\d+(\.\d+)*$", stripped):
        return f"The text states that the exact value is {stripped}."
    _DATE_WORDS = {"monday", "tuesday", "wednesday", "thursday", "friday",
                   "saturday", "sunday", "january", "february", "march",
                   "april", "may", "june", "july", "august", "september",
                   "october", "november", "december", "am", "pm"}
    lower = stripped.lower()
    # NOTE: substring test, so "am"/"pm"/"may" match inside ordinary words; almost every claim takes this branch.
    if any(w in lower for w in _DATE_WORDS) or re.search(r"\d", stripped):
        return f"The text explicitly states the time or date as {stripped}."
    if len(stripped.split()) <= 2:
        return f"The text explicitly mentions {stripped}."
    return f"Specifically, {stripped}."

def premise_windows(premise: str, window=WINDOW, overlap=OVERLAP):
    # Port of server.pipeline.entailment._premise_windows
    words = premise.split()
    if len(words) <= window:
        return [premise]
    step = window - overlap
    return [" ".join(words[s:s + window]) for s in range(0, len(words), step) if words[s:s + window]]

def build_pairs(docs, claims, mode="windowed", hypothesis_mode="repo", window=WINDOW, overlap=OVERLAP):
    # Flatten to (record_idx, premise_text, hypothesis_text). Empty claims are skipped (repo returns score 0.0).
    rec, prem, hyp = [], [], []
    for i, (d, c) in enumerate(zip(docs, claims)):
        if not c or not c.strip():
            continue
        h = make_hypothesis(c) if hypothesis_mode == "repo" else c.strip()
        for w in (premise_windows(d, window, overlap) if mode == "windowed" else [d]):
            rec.append(i); prem.append(w); hyp.append(h)
    return np.asarray(rec, dtype=np.int64), prem, hyp

# ---- Model -----------------------------------------------------------------------------------
def load_model(name=MODEL_NAME, path=MODEL_PATH, device=DEVICE, fp16=True):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    src = path or name
    tok = AutoTokenizer.from_pretrained(src)
    model = AutoModelForSequenceClassification.from_pretrained(src).eval()
    id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
    entail_idx = [k for k, v in id2label.items() if v == "entailment"]
    assert len(entail_idx) == 1, f"Could not find a unique 'entailment' label in {id2label}"
    if fp16 and device == "cuda":
        model = model.half()
    return tok, model.to(device), entail_idx[0], id2label

# ---- Batched inference -----------------------------------------------------------------------
def _tokenize(tok, prem, hyp, max_len, chunk=4096):
    ids = []
    for s in range(0, len(prem), chunk):
        enc = tok(prem[s:s + chunk], hyp[s:s + chunk], truncation=True, max_length=max_len,
                  add_special_tokens=True, return_attention_mask=False, return_token_type_ids=False)
        ids.extend(enc["input_ids"])
    return ids

def _make_batches(lengths, token_budget, max_batch):
    order = np.argsort(-lengths, kind="stable")          # longest first -> OOM shows up immediately
    batches, i = [], 0
    while i < len(order):
        bs = int(max(1, min(max_batch, token_budget // max(1, lengths[order[i]]))))
        batches.append(order[i:i + bs]); i += bs
    return batches

def _save_ckpt(path, scores, sig):
    tmp = path + ".tmp.npz"
    np.savez(tmp, scores=scores, sig=np.array(sig))
    os.replace(tmp, path)

@torch.inference_mode()
def score_pairs(tok, model, entail_idx, prem, hyp, ckpt_path=None, sig="", max_len=MAX_LEN,
                token_budget=TOKEN_BUDGET, max_batch=MAX_BATCH, desc="scoring"):
    # Returns float32 entailment probability per (premise, hypothesis) pair.
    n = len(prem)
    scores = np.full(n, np.nan, dtype=np.float32)
    if n == 0:
        return scores
    if ckpt_path and os.path.exists(ckpt_path):
        with np.load(ckpt_path, allow_pickle=False) as z:
            ck_sig, ck_scores = str(z["sig"]), z["scores"].copy()
        if ck_sig == sig and ck_scores.shape == scores.shape:
            scores = ck_scores
            print(f"[{desc}] resumed from checkpoint: {int((~np.isnan(scores)).sum())}/{n} pairs already done")
    ids = _tokenize(tok, prem, hyp, max_len)
    lengths = np.asarray([len(x) for x in ids])
    pad_id = tok.pad_token_id or 0
    batches = _make_batches(lengths, token_budget, max_batch)
    todo = [b for b in batches if np.isnan(scores[b]).any()]
    dev = next(model.parameters()).device
    last_ckpt = time.time()
    bar = tqdm(total=int(sum(len(b) for b in todo)), desc=desc, unit="pair")
    k = 0
    while k < len(todo):
        b = todo[k]
        try:
            L = int(lengths[b].max())
            x = torch.full((len(b), L), pad_id, dtype=torch.long)
            m = torch.zeros((len(b), L), dtype=torch.long)
            for r, j in enumerate(b):
                t = ids[j]; x[r, :len(t)] = torch.tensor(t); m[r, :len(t)] = 1
            logits = model(input_ids=x.to(dev), attention_mask=m.to(dev)).logits.float()
            scores[b] = torch.softmax(logits, dim=-1)[:, entail_idx].cpu().numpy()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(b) == 1:
                raise
            half = len(b) // 2                         # split this batch and retry; shrink future batches too
            todo[k:k + 1] = [b[:half], b[half:]]
            token_budget = max(512, token_budget // 2)
            print(f"[{desc}] OOM -> splitting batch, token budget now {token_budget}")
            continue
        bar.update(len(b)); k += 1
        if ckpt_path and time.time() - last_ckpt > CKPT_EVERY_SEC:
            _save_ckpt(ckpt_path, scores, sig); last_ckpt = time.time()
    bar.close()
    if ckpt_path and todo:
        _save_ckpt(ckpt_path, scores, sig)
    assert not np.isnan(scores).any()
    return scores

def score_records(tok, model, entail_idx, docs, claims, mode="windowed", hypothesis_mode="repo",
                  ckpt_path=None, tag="", desc="scoring", **kw):
    # Per-record score = max entailment over its premise windows (0.0 if the claim is empty, as in the repo).
    rec, prem, hyp = build_pairs(docs, claims, mode, hypothesis_mode)
    sig = hashlib.md5(json.dumps([tag, mode, hypothesis_mode, WINDOW, OVERLAP, MAX_LEN, len(docs), len(prem),
                                  hashlib.md5("".join(hyp[:50]).encode()).hexdigest()]).encode()).hexdigest()
    print(f"[{desc}] {len(docs)} records -> {len(prem)} (window, claim) pairs")
    ps = score_pairs(tok, model, entail_idx, prem, hyp, ckpt_path=ckpt_path, sig=sig, desc=desc, **kw)
    out = np.zeros(len(docs), dtype=np.float64)
    np.maximum.at(out, rec, ps.astype(np.float64))
    return out

## 4. Metrics

In [ ]:
def confusion(y, pred):
    y = np.asarray(y, bool); pred = np.asarray(pred, bool)
    return dict(tp=int((pred & y).sum()), fp=int((pred & ~y).sum()), tn=int((~pred & ~y).sum()), fn=int((~pred & y).sum()))

def basic_metrics(y, pred):
    c = confusion(y, pred); tp, fp, tn, fn = c["tp"], c["fp"], c["tn"], c["fn"]
    tot = tp + fp + tn + fn
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    return {"accuracy": (tp + tn) / tot if tot else 0.0, "balanced_accuracy": 0.5 * (R + spec),
            "precision": P, "recall": R, "f1": 2 * P * R / (P + R) if P + R else 0.0, **c, "total": tot}

def _rank_avg(x):
    # average ranks (1-based) with tie handling, numpy only
    o = np.argsort(x, kind="mergesort"); xs = x[o]
    r = np.empty(len(x), dtype=np.float64)
    _, start, cnt = np.unique(xs, return_index=True, return_counts=True)
    avg = start + (cnt + 1) / 2.0
    r[o] = np.repeat(avg, cnt)
    return r

def auroc(y, s):
    y = np.asarray(y, bool); s = np.asarray(s, float)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return float("nan")
    return float((_rank_avg(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def auprc(y, s):
    # average precision (positive class = supported), ties grouped
    y = np.asarray(y, bool); s = np.asarray(s, float)
    if y.sum() == 0:
        return float("nan")
    o = np.argsort(-s, kind="mergesort"); ys = y[o]; ss = s[o]
    tp = np.cumsum(ys); fp = np.cumsum(~ys)
    last = np.r_[ss[1:] != ss[:-1], True]               # last index of each tie group
    tp, fp = tp[last], fp[last]
    prec = tp / (tp + fp); rec = tp / y.sum()
    return float(np.sum(np.diff(np.r_[0.0, rec]) * prec))

def full_metrics(y, s, thr):
    y = np.asarray(y, bool); s = np.asarray(s, float)
    m = basic_metrics(y, s >= thr)
    m.update(auroc=auroc(y, s), auprc=auprc(y, s), threshold=float(thr))
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()}

def calibrate_threshold(y, s, grid=None):
    # Pick the threshold maximising balanced accuracy (ties -> lowest balanced-acc-equal threshold nearest 0.5).
    y = np.asarray(y, bool); s = np.asarray(s, float)
    grid = np.linspace(0.0, 1.0, 1001) if grid is None else grid
    ss1, ss0 = np.sort(s[y]), np.sort(s[~y])
    tpr = 1 - np.searchsorted(ss1, grid, side="left") / max(1, len(ss1))      # P(score >= t | supported)
    tnr = np.searchsorted(ss0, grid, side="left") / max(1, len(ss0))          # P(score <  t | unsupported)
    ba = 0.5 * (tpr + tnr)
    best = np.flatnonzero(ba >= ba.max() - 1e-12)
    i = best[np.argmin(np.abs(grid[best] - 0.5))]
    return float(grid[i]), float(ba[i]), grid, ba

def bootstrap_ci(y, s, thr, n_boot=N_BOOT, seed=SEED):
    y = np.asarray(y, bool); s = np.asarray(s, float); rng = np.random.default_rng(seed)
    ba, au = [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        yb, sb = y[i], s[i]
        if yb.all() or (~yb).all():
            continue
        ba.append(basic_metrics(yb, sb >= thr)["balanced_accuracy"]); au.append(auroc(yb, sb))
    q = lambda a: [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
    return {"balanced_accuracy_95ci": q(ba), "auroc_95ci": q(au), "n_boot": len(ba)}

def per_source(df, s, thr):
    out = {}
    if "dataset" not in df.columns:
        return out
    for name, g in df.assign(_s=s).groupby("dataset"):
        y = g["label"].astype(bool).values
        m = basic_metrics(y, g["_s"].values >= thr)
        out[name] = {"n": int(len(g)), "frac_supported": round(float(y.mean()), 4),
                     "accuracy": round(m["accuracy"], 4), "balanced_accuracy": round(m["balanced_accuracy"], 4),
                     "precision": round(m["precision"], 4), "recall": round(m["recall"], 4),
                     "auroc": round(auroc(y, g["_s"].values), 4)}
    return out

def stratified_sample(df, n, seed=SEED):
    # seeded sample, proportional within (dataset, label) strata; keeps original row order
    if n is None or n >= len(df):
        return df
    keys = [c for c in ("dataset", "label") if c in df.columns]
    frac = n / len(df)
    parts = [g.sample(n=max(1, round(len(g) * frac)), random_state=seed)
             for _, g in df.groupby(keys, sort=True)]    # loop (not .apply) keeps the grouping columns in every pandas version
    out = pd.concat(parts)
    if len(out) > n:                                    # tiny n: per-stratum minimum of 1 can overshoot
        out = out.sample(n=n, random_state=seed)
    return out.sort_index()

## 5. Load data and model

In [ ]:
def load_split(split, limit):
    df = pd.concat([pd.read_parquet(p) for p in PATHS[split]], ignore_index=True)
    missing = {"doc", "claim", "label"} - set(df.columns)
    assert not missing, f"missing columns {missing}; got {list(df.columns)}"
    df["row_id"] = [f"local:{split}:{i}" for i in range(len(df))]       # same ids as the repo's evaluator (pre-sampling index)
    df["label"] = df["label"].astype(int)
    full_n, full_pos = len(df), df["label"].mean()
    df = stratified_sample(df, limit).reset_index(drop=True)
    print(f"{split}: {full_n} rows in file, using {len(df)} | supported fraction {df['label'].mean():.4f} (file: {full_pos:.4f})")
    return df

dev_df  = load_split("dev",  DEV_LIMIT)
test_df = load_split("test", TEST_LIMIT)
print("columns:", list(test_df.columns))
print("words/doc (test) mean / p95 / max:", int(test_df.doc.str.split().str.len().mean()),
      int(test_df.doc.str.split().str.len().quantile(.95)), int(test_df.doc.str.split().str.len().max()))

def autodetect_model_path():
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if any(os.path.exists(os.path.join(d, f)) for f in ("model.safetensors", "pytorch_model.bin")):
            try:
                if "deberta" in json.load(open(cfg)).get("model_type", ""):
                    return d
            except Exception:
                pass
    return None

_mp = MODEL_PATH or autodetect_model_path()
print("model source:", _mp or f"Hugging Face hub ({MODEL_NAME})")
try:
    tok, model, ENTAIL_IDX, ID2LABEL = load_model(path=_mp)
except Exception as e:
    raise RuntimeError("Could not load the NLI model. Either set Internet = On in the notebook settings "
                       "(needs a phone-verified Kaggle account) or attach the 'deberta-nli' dataset as input.") from e
print("label map:", ID2LABEL, "-> entailment index", ENTAIL_IDX)

## 6. Score DEV, calibrate threshold, then score TEST once
Scores are checkpointed to `/kaggle/working/ckpt_*.npz`; if the session dies, re-run the notebook and it resumes.

In [ ]:
t0 = time.time()
tag = f"{HYPOTHESIS_MODE}-{WINDOW}-{OVERLAP}-{MAX_LEN}"
dev_scores = score_records(tok, model, ENTAIL_IDX, dev_df.doc.tolist(), dev_df.claim.tolist(),
                           hypothesis_mode=HYPOTHESIS_MODE, ckpt_path=f"{OUT_DIR}/ckpt_dev_{tag}_{DEV_LIMIT}.npz",
                           tag="dev", desc="dev")
print(f"dev scored in {time.time()-t0:.0f}s")

y_dev = dev_df.label.values.astype(bool)
THR, dev_ba, grid, ba_curve = calibrate_threshold(y_dev, dev_scores)
print(f"Calibrated threshold (DEV, max balanced acc): {THR:.3f}  | DEV balanced acc = {dev_ba:.4f}")
print(f"DEV at fixed 0.40: balanced acc = {basic_metrics(y_dev, dev_scores >= FIXED_THRESHOLD)['balanced_accuracy']:.4f}")

t1 = time.time()
test_scores = score_records(tok, model, ENTAIL_IDX, test_df.doc.tolist(), test_df.claim.tolist(),
                            hypothesis_mode=HYPOTHESIS_MODE, ckpt_path=f"{OUT_DIR}/ckpt_test_{tag}_{TEST_LIMIT}.npz",
                            tag="test", desc="test")
print(f"test scored in {time.time()-t1:.0f}s")

## 7. Results (ENTAILMENT-ONLY, windowed)

In [ ]:
y_test = test_df.label.values.astype(bool)
res_cal   = full_metrics(y_test, test_scores, THR)
res_fixed = full_metrics(y_test, test_scores, FIXED_THRESHOLD)
ci_cal    = bootstrap_ci(y_test, test_scores, THR)
ci_fixed  = bootstrap_ci(y_test, test_scores, FIXED_THRESHOLD)
maj_acc   = max(y_test.mean(), 1 - y_test.mean())
majority  = {"predict": "supported" if y_test.mean() >= .5 else "unsupported", "accuracy": round(float(maj_acc), 4),
             "balanced_accuracy": 0.5, "frac_supported": round(float(y_test.mean()), 4)}

def show(title, m, ci):
    print(f"--- {title} (threshold={m['threshold']:.3f}) ---")
    for k in ("accuracy", "balanced_accuracy", "precision", "recall", "f1", "auroc", "auprc"):
        print(f"  {k:<18}: {m[k]:.4f}")
    print(f"  bal-acc 95% CI    : {ci['balanced_accuracy_95ci']}   AUROC 95% CI: {ci['auroc_95ci']}  (n_boot={ci['n_boot']})")
    print(f"  confusion         : TP={m['tp']}  FP={m['fp']}  TN={m['tn']}  FN={m['fn']}")
    print("                      pred_unsupp  pred_supp")
    print(f"  true unsupported  : {m['tn']:>10} {m['fp']:>10}")
    print(f"  true supported    : {m['fn']:>10} {m['tp']:>10}")

print(f"ENTAILMENT-ONLY (no GED) | n={len(test_df)} | windowed {WINDOW}/{OVERLAP} | hypothesis_mode={HYPOTHESIS_MODE}")
show("TEST @ DEV-calibrated threshold", res_cal, ci_cal)
show("TEST @ fixed 0.40", res_fixed, ci_fixed)
print(f"\nMajority-class baseline (always '{majority['predict']}'): accuracy={majority['accuracy']:.4f}, balanced acc=0.5000")

src_cal = per_source(test_df, test_scores, THR)
if src_cal:
    print("\nPer-source (threshold = DEV-calibrated):")
    print(pd.DataFrame(src_cal).T.to_string())
else:
    print("\nNo 'dataset' column: per-source breakdown skipped.")

## 8. Comparison with the old truncation baseline

In [ ]:
trunc_res = None
if RUN_TRUNC_BASELINE:
    # Same hypotheses and model, but one pair per record with the whole document truncated at MAX_LEN tokens.
    sub = stratified_sample(test_df, TRUNC_SAMPLE).copy()
    idx = sub.index.values
    trunc_scores = score_records(tok, model, ENTAIL_IDX, sub.doc.tolist(), sub.claim.tolist(), mode="truncate",
                                 hypothesis_mode=HYPOTHESIS_MODE, ckpt_path=f"{OUT_DIR}/ckpt_trunc_{tag}_{TRUNC_SAMPLE}.npz",
                                 tag="trunc", desc="trunc-baseline")
    ys = sub.label.values.astype(bool)
    trunc_res = full_metrics(ys, trunc_scores, FIXED_THRESHOLD)
    win_same = full_metrics(ys, test_scores[idx], FIXED_THRESHOLD)
    print(f"Same {len(sub)}-row test sample @ 0.40:   truncation vs windowed")
    for k in ("accuracy", "balanced_accuracy", "precision", "recall", "f1", "auroc"):
        print(f"  {k:<18} {trunc_res[k]:.4f}  ->  {win_same[k]:.4f}")

rows = {"old truncation (full test, STATUS.md, thr 0.40)": OLD_BASELINE,
        "windowed @ 0.40": res_fixed, "windowed @ DEV-calibrated": res_cal}
if trunc_res:
    rows[f"truncation recomputed ({TRUNC_SAMPLE}-row sample, 0.40)"] = trunc_res
comp = pd.DataFrame({k: {m: v.get(m) for m in ("threshold", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "auroc")}
                     for k, v in rows.items()}).T
print("\n", comp.round(4).to_string())
print(f"\nDelta balanced accuracy (windowed@DEV-cal vs old): {res_cal['balanced_accuracy'] - OLD_BASELINE['balanced_accuracy']:+.4f}")
print("Caveat: the old baseline is the full 29,320-row test run; if TEST_LIMIT is set this is a different sample.")

### 8b. Hypothesis-template ablation
The repo's `_make_hypothesis` rewrites almost every claim as *"The text explicitly states the time or date as ..."* (substring date test). This cell scores a 2k test sample with that template to show its effect versus the raw claim.

In [ ]:
if RUN_TEMPLATE_ABLATION:
    other = "repo" if HYPOTHESIS_MODE == "raw" else "raw"
    sub2 = stratified_sample(test_df, TRUNC_SAMPLE).copy()
    sc_other = score_records(tok, model, ENTAIL_IDX, sub2.doc.tolist(), sub2.claim.tolist(),
                             hypothesis_mode=other, ckpt_path=f"{OUT_DIR}/ckpt_abl_{other}_{WINDOW}_{OVERLAP}_{TRUNC_SAMPLE}.npz",
                             tag="abl", desc=f"ablation-{other}")
    ys2 = sub2.label.values.astype(bool)
    sc_main = test_scores[sub2.index.values]
    print(f"Same {len(sub2)}-row sample, windowed, @ 0.40:   {HYPOTHESIS_MODE}  vs  {other}")
    for k, fn in (("balanced_accuracy", None), ("auroc", None), ("recall", None), ("precision", None)):
        a = full_metrics(ys2, sc_main, FIXED_THRESHOLD)[k]; b = full_metrics(ys2, sc_other, FIXED_THRESHOLD)[k]
        print(f"  {k:<18} {a:.4f}  vs  {b:.4f}")


## 8c. Three-way baseline comparison incl. MiniCheck (FACT-CHECK / NLI SCORERS ONLY, no GED)
Fair comparison on the **same TEST rows**, each system with its own threshold tuned on DEV (max balanced accuracy) and evaluated once on TEST:

| | system | notes |
|---|---|---|
| A | DeBERTa-v3-base NLI, **truncation** (whole doc cut at 512 tokens) | the "old" behaviour, re-scored here on the full DEV and TEST rows (one pair per row, cheap) |
| B | DeBERTa-v3-base NLI, **windowed** 180/40 words, max over windows | the scores computed above |
| C | **MiniCheck** (published fact-checker, Tang et al., EMNLP 2024) | uses its own sentence-based chunking of long documents (default chunk size 400 tokens for roberta/deberta-large, 500 for flan-t5-large) and max over chunks; we do **not** window it ourselves |

All three are document-vs-claim fact-check scorers only. None includes the GED structural comparison, so none of these numbers describe the full pipeline.
MiniCheck is a published baseline: whatever numbers come out are reported as they are.

Settings: MiniCheck needs **Internet = On** (pip install from GitHub + model download from Hugging Face). Set `RUN_MINICHECK = False` below to skip it; rows A and B are still produced.

In [ ]:
# ---------------- MiniCheck / comparison config ----------------
RUN_COMPARISON = True        # build the A/B/C table (A needs a full-split truncation pass; cheap relative to the windowed run)
RUN_MINICHECK  = True        # False = skip MiniCheck entirely (no install, no download); table then has A and B only
MINICHECK_MODEL = "flan-t5-large"   # one of 'flan-t5-large' (770M, best <1B per the MiniCheck README; ~3 GB fp32 -> fits a T4 16 GB),
                                    #  'deberta-v3-large' (0.4B), 'roberta-large' (0.4B, 512 max len).
                                    # NOT 'Bespoke-MiniCheck-7B' / 'Granite-Guardian-*': they need vLLM (Linux) and far more VRAM than a T4 offers.
MINICHECK_BATCH = 4          # `batch_size` argument of MiniCheck(...) = number of document chunks scored per forward pass (package default 16)
MINICHECK_CHUNK_SIZE = None  # None = package default (400 / 500 tokens); an int overrides `chunk_size` in .score()
MINICHECK_BLOCK = 250        # rows per checkpoint block (a crash loses at most one block)
MINICHECK_CACHE_DIR = "/kaggle/working/minicheck_ckpts" if os.path.isdir("/kaggle/working") else "./minicheck_ckpts"
MINICHECK_PIP = "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"   # install command exactly as in the official README
TRUNC_HYPOTHESIS_MODE = None # hypothesis mode for system A; None = same as HYPOTHESIS_MODE (isolates the truncation-vs-windowing effect).
                             # Set "repo" to mimic the old baseline's templated claims exactly (then A vs B differs in template AND windowing).
N_BOOT_PAIRED = 1000         # bootstrap resamples for the comparison CIs and the paired bootstrap
# MiniCheck is ~10x slower than DeBERTa (about 8 h for the full DEV+TEST on a T4) and ran out of GPU memory on long documents at batch 16.
# So it is scored on a seeded, label/source-stratified SUBSET (None = full split). A, B and C are all compared on exactly this subset.
MINICHECK_DEV_N  = 3000      # DEV rows used to tune the thresholds for the subset table
MINICHECK_TEST_N = 5000      # TEST rows used for the three-way comparison


### 8c.1 Install MiniCheck and load the model
Install command and API are copied from the official README (`github.com/Liyan06/MiniCheck`):
`pip install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"`, then
`from minicheck.minicheck import MiniCheck; scorer = MiniCheck(model_name=..., cache_dir=...)` and
`pred_label, raw_prob, used_chunks, prob_per_chunk = scorer.score(docs=[...], claims=[...])`; `raw_prob` is the max "supported" probability over chunks.
`batch_size=` is a documented constructor argument of the package source (`minicheck/minicheck.py`).

In [ ]:
MC_AVAILABLE = False
MC_ERROR = None
mc_scorer = None

def _skip_msg(stage, err):
    return (f"\n*** MiniCheck {stage} FAILED: {type(err).__name__}: {err}\n"
            f"*** Likely causes: Internet is Off in the notebook settings, no phone-verified account, or GitHub / Hugging Face unreachable.\n"
            f"*** The rest of the notebook is unaffected. To skip MiniCheck cleanly set RUN_MINICHECK = False in the config cell and re-run;\n"
            f"*** the comparison table is then produced for A (truncation) and B (windowed) only.\n")

if RUN_COMPARISON and RUN_MINICHECK:
    try:
        try:
            import minicheck  # noqa: F401
        except ImportError:
            print("Installing MiniCheck (needs Internet = On) ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", MINICHECK_PIP])
        # MiniCheck splits documents with nltk.sent_tokenize; the README does not mention fetching tokenizer data.
        # UNVERIFIED: the punkt downloads below are my addition (older nltk needs 'punkt', newer needs 'punkt_tab').
        import nltk
        for _r in ("punkt", "punkt_tab"):
            try:
                nltk.download(_r, quiet=True)
            except Exception as _e:
                print(f"nltk.download({_r!r}) failed (continuing): {_e}")
        nltk.sent_tokenize("Smoke test one. Smoke test two.")      # fail early here if tokenizer data is missing
        from minicheck.minicheck import MiniCheck
        os.makedirs(MINICHECK_CACHE_DIR, exist_ok=True)
        mc_scorer = MiniCheck(model_name=MINICHECK_MODEL, batch_size=MINICHECK_BATCH, cache_dir=MINICHECK_CACHE_DIR)
        # tiny functional check, same example as the README (expected roughly [1, 0]; exact probabilities vary with hardware)
        _d = "A group of students gather in the school library to study for their upcoming final exams."
        _lab, _p, _, _ = mc_scorer.score(docs=[_d, _d], claims=["The students are preparing for an examination.", "The students are on vacation."])
        print("MiniCheck smoke test (README example, expect ~[1, 0]):", list(_lab), [round(float(x), 4) for x in _p])
        MC_AVAILABLE = True
    except Exception as e:
        MC_ERROR = f"{type(e).__name__}: {e}"
        print(_skip_msg("install/load", e))
else:
    print("MiniCheck skipped (RUN_MINICHECK=False or RUN_COMPARISON=False).")

### 8c.2 Score DEV and TEST with MiniCheck (checkpointed) and the truncation baseline on the full splits
MiniCheck is run on the raw `(doc, claim)` pairs in blocks; each finished block is written to `ckpt_minicheck_*.npz`, so a crash resumes from the last block.
The truncation baseline (A) reuses `score_records(..., mode="truncate")` and its checkpointing.

In [ ]:
def minicheck_scores(scorer, docs, claims, ckpt_path, tag, block=MINICHECK_BLOCK, chunk_size=MINICHECK_CHUNK_SIZE, desc="minicheck"):
    # Returns float64 array of MiniCheck raw "supported" probabilities (max over its own chunks), resumable per block.
    n = len(docs)
    sig = hashlib.md5(json.dumps([tag, MINICHECK_MODEL, chunk_size, n,
                                  hashlib.md5("".join(claims[:50]).encode()).hexdigest()]).encode()).hexdigest()
    scores = np.full(n, np.nan, dtype=np.float64)
    if os.path.exists(ckpt_path):
        with np.load(ckpt_path, allow_pickle=False) as z:
            if str(z["sig"]) == sig and z["scores"].shape == scores.shape:
                scores = z["scores"].astype(np.float64).copy()
                print(f"[{desc}] resumed: {int((~np.isnan(scores)).sum())}/{n} rows already scored")
    t0 = time.time()
    for s in range(0, n, block):
        e = min(n, s + block)
        if not np.isnan(scores[s:e]).any():
            continue
        kw = {"chunk_size": chunk_size} if chunk_size else {}
        _, prob, _, _ = scorer.score(docs=list(docs[s:e]), claims=list(claims[s:e]), **kw)
        prob = np.asarray(prob, dtype=np.float64)
        assert prob.shape == (e - s,), f"unexpected MiniCheck output shape {prob.shape}"
        scores[s:e] = prob
        _save_ckpt(ckpt_path, scores, sig)
        print(f"[{desc}] {int((~np.isnan(scores)).sum())}/{n} rows | {time.time()-t0:.0f}s elapsed")
    assert not np.isnan(scores).any()
    return scores

cmp_scores = {}     # system key -> {"dev": array, "test": array}
cmp_names = {"A": "A: DeBERTa-v3-base, truncation (512 tok)",
             "B": "B: DeBERTa-v3-base, windowed 180/40",
             "C": f"C: MiniCheck ({MINICHECK_MODEL})"}

if RUN_COMPARISON:
    # --- A: truncation baseline on the full DEV and TEST rows of this run
    _thm = TRUNC_HYPOTHESIS_MODE or HYPOTHESIS_MODE
    _ta = f"{_thm}-trunc-{MAX_LEN}"
    t0 = time.time()
    a_dev = score_records(tok, model, ENTAIL_IDX, dev_df.doc.tolist(), dev_df.claim.tolist(), mode="truncate", hypothesis_mode=_thm,
                          ckpt_path=f"{OUT_DIR}/ckpt_cmpA_dev_{_ta}_{DEV_LIMIT}.npz", tag="cmpA-dev", desc="A dev (trunc)")
    a_test = score_records(tok, model, ENTAIL_IDX, test_df.doc.tolist(), test_df.claim.tolist(), mode="truncate", hypothesis_mode=_thm,
                           ckpt_path=f"{OUT_DIR}/ckpt_cmpA_test_{_ta}_{TEST_LIMIT}.npz", tag="cmpA-test", desc="A test (trunc)")
    print(f"truncation baseline scored in {time.time()-t0:.0f}s")
    cmp_scores["A"] = {"dev": a_dev, "test": a_test}
    cmp_scores["B"] = {"dev": dev_scores, "test": test_scores}

    # --- C: MiniCheck on a seeded stratified subset (same rows are used to compare A, B and C)
    if MC_AVAILABLE:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            idx_dev = np.sort((stratified_sample(dev_df, MINICHECK_DEV_N) if MINICHECK_DEV_N else dev_df).index.values)
            idx_test = np.sort((stratified_sample(test_df, MINICHECK_TEST_N) if MINICHECK_TEST_N else test_df).index.values)
            t0 = time.time()
            _tc = f"{MINICHECK_MODEL}_{MINICHECK_CHUNK_SIZE}_b{MINICHECK_BATCH}"
            c_dev = minicheck_scores(mc_scorer, dev_df.doc.values[idx_dev].tolist(), dev_df.claim.values[idx_dev].tolist(),
                                     f"{OUT_DIR}/ckpt_minicheck_dev_{_tc}_{len(idx_dev)}.npz", "mc-dev", desc="MiniCheck dev")
            print(f"MiniCheck dev scored in {time.time()-t0:.0f}s"); t1 = time.time()
            c_test = minicheck_scores(mc_scorer, test_df.doc.values[idx_test].tolist(), test_df.claim.values[idx_test].tolist(),
                                      f"{OUT_DIR}/ckpt_minicheck_test_{_tc}_{len(idx_test)}.npz", "mc-test", desc="MiniCheck test")
            print(f"MiniCheck test scored in {time.time()-t1:.0f}s")
            cmp_scores["C"] = {"dev": c_dev, "test": c_test, "dev_idx": idx_dev, "test_idx": idx_test}
        except Exception as e:
            MC_ERROR = f"{type(e).__name__}: {e}"
            print(_skip_msg("scoring", e), "(finished block checkpoints are kept; re-run this cell to resume)")


### 8c.3 Paired bootstrap and comparison table
`paired_bootstrap` resamples TEST rows once and evaluates both systems on the **same** resample (each at its own DEV-tuned threshold), so the CI of the difference reflects the pairing.
The reported p is a two-sided bootstrap p-value, `2 * min(P(diff <= 0), P(diff >= 0))`, floored at `1/n_boot`; treat it as indicative, not exact.

In [ ]:
def paired_bootstrap(y, s_a, s_b, thr_a, thr_b, n_boot=1000, seed=42):
    # Difference (a - b) in balanced accuracy and AUROC, resampling rows jointly. Pure numpy; reuses basic_metrics / auroc above.
    y = np.asarray(y, bool); s_a = np.asarray(s_a, float); s_b = np.asarray(s_b, float)
    assert len(y) == len(s_a) == len(s_b)
    rng = np.random.default_rng(seed)
    d_ba, d_au = [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        yb = y[i]
        if yb.all() or (~yb).all():
            continue
        ba_a = basic_metrics(yb, s_a[i] >= thr_a)["balanced_accuracy"]; ba_b = basic_metrics(yb, s_b[i] >= thr_b)["balanced_accuracy"]
        d_ba.append(ba_a - ba_b); d_au.append(auroc(yb, s_a[i]) - auroc(yb, s_b[i]))
    def summ(d, point):
        d = np.asarray(d)
        p = max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1.0 / max(1, len(d)))
        return {"diff": round(float(point), 4), "ci95": [round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)],
                "p_two_sided": round(float(min(1.0, p)), 4)}
    pt_ba = basic_metrics(y, s_a >= thr_a)["balanced_accuracy"] - basic_metrics(y, s_b >= thr_b)["balanced_accuracy"]
    pt_au = auroc(y, s_a) - auroc(y, s_b)
    return {"balanced_accuracy": summ(d_ba, pt_ba), "auroc": summ(d_au, pt_au), "n_boot": len(d_ba)}

comparison = None
if RUN_COMPARISON:
    y_dev_c = dev_df.label.values.astype(bool); y_test_c = test_df.label.values.astype(bool)
    systems = {}
    for k, sc in cmp_scores.items():
        if k == "C":
            continue          # MiniCheck is scored on a subset; it is compared in the subset table below
        thr_k, dev_ba_k, _, _ = calibrate_threshold(y_dev_c, sc["dev"])       # tuned on DEV only
        systems[k] = {"name": cmp_names[k], "threshold": thr_k, "dev_balanced_accuracy": round(dev_ba_k, 4),
                      "test_metrics": full_metrics(y_test_c, sc["test"], thr_k),            # evaluated once on TEST
                      "test_ci": bootstrap_ci(y_test_c, sc["test"], thr_k, n_boot=N_BOOT_PAIRED)}
    paired = {}
    if "C" in systems:
        paired["B_minus_C"] = paired_bootstrap(y_test_c, cmp_scores["B"]["test"], cmp_scores["C"]["test"],
                                               systems["B"]["threshold"], systems["C"]["threshold"], n_boot=N_BOOT_PAIRED)
    paired["B_minus_A"] = paired_bootstrap(y_test_c, cmp_scores["B"]["test"], cmp_scores["A"]["test"],
                                           systems["B"]["threshold"], systems["A"]["threshold"], n_boot=N_BOOT_PAIRED)
    per_src_cmp = {k: per_source(test_df, cmp_scores[k]["test"], systems[k]["threshold"]) for k in systems}
    comparison = {"protocol": "per-system threshold tuned on DEV (max balanced accuracy); evaluated once on the same TEST rows; "
                              "bootstrap over TEST rows; all systems are fact-check/NLI scorers only (no GED)",
                  "n_test": int(len(test_df)), "n_dev": int(len(dev_df)), "systems": systems, "paired_bootstrap": paired,
                  "minicheck_status": ("ok" if "C" in cmp_scores else ("skipped" if not RUN_MINICHECK else f"failed: {MC_ERROR}")),
                  "per_source": per_src_cmp}

    rows = []
    for k, s in systems.items():
        m, ci = s["test_metrics"], s["test_ci"]
        rows.append({"system": s["name"], "thr (DEV)": round(s["threshold"], 3), "DEV bal.acc": s["dev_balanced_accuracy"],
                     "TEST bal.acc": m["balanced_accuracy"], "bal.acc 95% CI": ci["balanced_accuracy_95ci"],
                     "AUROC": m["auroc"], "AUROC 95% CI": ci["auroc_95ci"], "precision": m["precision"], "recall": m["recall"], "F1": m["f1"]})
    cmp_df = pd.DataFrame(rows).set_index("system")
    print(cmp_df.to_string())
    for name, r in paired.items():
        print(f"\npaired bootstrap {name} ({r['n_boot']} resamples):")
        for met in ("balanced_accuracy", "auroc"):
            print(f"  {met:<18} diff={r[met]['diff']:+.4f}  95% CI {r[met]['ci95']}  p~{r[met]['p_two_sided']}")
    if "C" not in systems:
        print(f"\n(MiniCheck row absent: {comparison['minicheck_status']})")

# ---- Three-way comparison on the MiniCheck subset (A, B and C scored on exactly the same DEV/TEST rows)
subset_df = None
if RUN_COMPARISON and "C" in cmp_scores:
    _c = cmp_scores["C"]; idd, idt = _c["dev_idx"], _c["test_idx"]
    y_d = y_dev_c[idd]; y_t = y_test_c[idt]
    sub, sub_test_scores = {}, {}
    for k in ("A", "B", "C"):
        sd = _c["dev"] if k == "C" else np.asarray(cmp_scores[k]["dev"])[idd]
        st = _c["test"] if k == "C" else np.asarray(cmp_scores[k]["test"])[idt]
        thr_k, dba, _, _ = calibrate_threshold(y_d, sd)
        sub_test_scores[k] = st
        sub[k] = {"name": cmp_names[k], "threshold": thr_k, "dev_balanced_accuracy": round(dba, 4),
                  "test_metrics": full_metrics(y_t, st, thr_k), "test_ci": bootstrap_ci(y_t, st, thr_k, n_boot=N_BOOT_PAIRED)}
    pairs = {"B_minus_C": ("B", "C"), "B_minus_A": ("B", "A"), "C_minus_A": ("C", "A")}
    paired_sub = {n: paired_bootstrap(y_t, sub_test_scores[a], sub_test_scores[b], sub[a]["threshold"], sub[b]["threshold"], n_boot=N_BOOT_PAIRED)
                  for n, (a, b) in pairs.items()}
    comparison["subset"] = {"n_dev": int(len(idd)), "n_test": int(len(idt)), "majority_baseline_accuracy": round(float(max(y_t.mean(), 1 - y_t.mean())), 4),
                            "systems": sub, "paired_bootstrap": paired_sub}
    rows = [{"system": s["name"], "thr (DEV subset)": round(s["threshold"], 4), "TEST bal.acc": s["test_metrics"]["balanced_accuracy"],
             "bal.acc 95% CI": s["test_ci"]["balanced_accuracy_95ci"], "AUROC": s["test_metrics"]["auroc"],
             "AUROC 95% CI": s["test_ci"]["auroc_95ci"], "precision": s["test_metrics"]["precision"], "recall": s["test_metrics"]["recall"],
             "F1": s["test_metrics"]["f1"]} for s in sub.values()]
    subset_df = pd.DataFrame(rows).set_index("system")
    print(f"\n=== Three-way comparison on the same subset: DEV n={len(idd)}, TEST n={len(idt)} ===")
    print(subset_df.to_string())
    for n, r in paired_sub.items():
        print(f"paired bootstrap {n}: bal.acc diff={r['balanced_accuracy']['diff']:+.4f} CI {r['balanced_accuracy']['ci95']} | "
              f"AUROC diff={r['auroc']['diff']:+.4f} CI {r['auroc']['ci95']}")


In [ ]:
from IPython.display import Markdown, display
if comparison:
    L = ["| system | threshold (DEV) | TEST bal. acc [95% CI] | TEST AUROC [95% CI] | precision | recall | F1 |", "|---|---|---|---|---|---|---|"]
    for k, s in comparison["systems"].items():
        m, ci = s["test_metrics"], s["test_ci"]
        L.append(f"| {s['name']} | {s['threshold']:.3f} | {m['balanced_accuracy']:.4f} {ci['balanced_accuracy_95ci']} | "
                 f"{m['auroc']:.4f} {ci['auroc_95ci']} | {m['precision']:.4f} | {m['recall']:.4f} | {m['f1']:.4f} |")
    PB = comparison["paired_bootstrap"]
    def _line(key, label):
        if key not in PB:
            return ""
        r = PB[key]
        return (f"- **{label}**: balanced accuracy {r['balanced_accuracy']['diff']:+.4f} (95% CI {r['balanced_accuracy']['ci95']}, p~{r['balanced_accuracy']['p_two_sided']}); "
                f"AUROC {r['auroc']['diff']:+.4f} (95% CI {r['auroc']['ci95']}, p~{r['auroc']['p_two_sided']})")
    mc_note = ("\n*MiniCheck was scored on a subset only; see the three-way table below.*\n" if comparison.get("subset") else
               f"\n**MiniCheck was not run / failed ({comparison['minicheck_status']}); only A and B are compared.**\n")
    table = chr(10).join(L)
    display(Markdown(f"""
### Comparison on the same {comparison['n_test']} LLM-AggreFact TEST rows (thresholds tuned on {comparison['n_dev']} DEV rows)

{table}

Paired bootstrap of the difference ({N_BOOT_PAIRED} resamples; positive = first system higher):

{_line('B_minus_C', 'windowed DeBERTa (B) minus MiniCheck (C)')}

{_line('B_minus_A', 'windowed DeBERTa (B) minus truncation DeBERTa (A)')}
{mc_note}
**Honest scope note.** All rows are document-vs-claim fact-check / NLI scorers only; the GED structural comparison of the full pipeline is not
part of this notebook, so none of this is a result for the full system. MiniCheck is a published baseline trained for exactly this task and is
reported as-is, with its own chunking (not our 180/40 windows). Each system's threshold was chosen on DEV and applied once to TEST; the CIs
resample TEST rows only (they do not capture threshold-selection or model-training variance). If TEST_LIMIT / DEV_LIMIT are set, the numbers
are for those stratified subsamples and are not comparable with published full-benchmark figures.
"""))

In [ ]:
from IPython.display import Markdown, display
if comparison and comparison.get("subset"):
    S_ = comparison["subset"]
    L_ = [f"### Three-way comparison on the same subset (DEV n={S_['n_dev']}, TEST n={S_['n_test']}; each threshold tuned on the DEV subset)", "",
          "| system | threshold | bal. acc [95% CI] | AUROC [95% CI] | precision | recall | F1 |", "|---|---|---|---|---|---|---|"]
    for s_ in S_["systems"].values():
        m_, ci_ = s_["test_metrics"], s_["test_ci"]
        L_.append(f"| {s_['name']} | {s_['threshold']:.4f} | {m_['balanced_accuracy']:.4f} {ci_['balanced_accuracy_95ci']} | {m_['auroc']:.4f} {ci_['auroc_95ci']} | {m_['precision']:.4f} | {m_['recall']:.4f} | {m_['f1']:.4f} |")
    L_ += ["", "Paired bootstrap of differences (positive = first system higher):"]
    for n_, r_ in S_["paired_bootstrap"].items():
        L_.append(f"- **{n_}**: balanced accuracy {r_['balanced_accuracy']['diff']:+.4f} (95% CI {r_['balanced_accuracy']['ci95']}), AUROC {r_['auroc']['diff']:+.4f} (95% CI {r_['auroc']['ci95']})")
    L_ += ["", "All three are fact-check/NLI scorers only (no GED). MiniCheck uses its own chunking and was scored on this subset only."]
    display(Markdown("\n".join(L_)))


## 9. Save results JSON and summary

In [ ]:
import platform
preds = [{
    "id": r.row_id, "prediction": bool(s >= THR), "ground_truth": bool(r.label),
    "entailment_score": round(float(s), 4), "source_dataset": getattr(r, "dataset", None),
    "hypothesis": r.claim, "premise": r.doc[:80] + "â€¦" if len(r.doc) > 80 else r.doc,
} for r, s in zip(test_df.itertuples(index=False), test_scores)]

results = {
    "source": {
        "source": "kaggle_local_parquet", "dataset_id": "lytang/LLM-AggreFact", "split": "test",
        "shards": PATHS["test"], "dev_shards": PATHS["dev"], "seed": SEED if TEST_LIMIT else None,
        "limit": TEST_LIMIT, "dev_limit": DEV_LIMIT, "rows_loaded": len(test_df), "dev_rows": len(dev_df),
        "license": "cc-by-nd-4.0", "dataset_url": "https://huggingface.co/datasets/lytang/LLM-AggreFact",
    },
    "evaluation": "ENTAILMENT-ONLY (DeBERTa NLI; GED not included)",
    "model": {"name": MODEL_PATH or MODEL_NAME, "id2label": ID2LABEL, "entailment_index": ENTAIL_IDX, "fp16": DEVICE == "cuda"},
    "scoring": {"mode": "windowed_max", "window_words": WINDOW, "overlap_words": OVERLAP, "max_len": MAX_LEN,
                "hypothesis_mode": HYPOTHESIS_MODE},
    "threshold": THR,
    "threshold_selection": {"method": "max balanced accuracy on DEV", "dev_balanced_accuracy": round(dev_ba, 4),
                            "fixed_threshold_compared": FIXED_THRESHOLD},
    "metrics": res_cal,
    "metrics_fixed_040": res_fixed,
    "confidence_intervals": {"at_calibrated_threshold": ci_cal, "at_fixed_040": ci_fixed},
    "majority_baseline": majority,
    "per_source": src_cal,
    "old_truncation_baseline": OLD_BASELINE,
    "truncation_recomputed": trunc_res,
    "runtime": {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "torch": torch.__version__, "transformers": transformers.__version__, "python": platform.python_version()},
    "predictions": preds,
}
# --- MiniCheck baseline and three-way comparison (section 8c); None when RUN_COMPARISON / RUN_MINICHECK are off or MiniCheck failed
results["minicheck"] = None
results["comparison"] = globals().get("comparison")
if globals().get("comparison") and "C" in comparison["systems"]:
    _c = comparison["systems"]["C"]
    results["minicheck"] = {"model": MINICHECK_MODEL, "package": MINICHECK_PIP, "batch_size": MINICHECK_BATCH,
                            "chunk_size": MINICHECK_CHUNK_SIZE or "package default (400; 500 for flan-t5-large)",
                            "threshold": _c["threshold"], "dev_balanced_accuracy": _c["dev_balanced_accuracy"],
                            "metrics": _c["test_metrics"], "confidence_intervals": _c["test_ci"],
                            "paired_bootstrap_B_minus_C": comparison["paired_bootstrap"].get("B_minus_C"),
                            "per_source": comparison["per_source"].get("C"), "scope": "fact-check scorer only, no GED"}
    results["minicheck_test_scores"] = [round(float(x), 4) for x in cmp_scores["C"]["test"]]
OUT_JSON = f"{OUT_DIR}/aggrefact_windowed_results.json"
with open(OUT_JSON, "w") as f:
    json.dump(results, f, indent=2, default=float)
print("saved", OUT_JSON, f"({os.path.getsize(OUT_JSON)/1e6:.1f} MB)")

from IPython.display import Markdown, display
display(Markdown(f'''
### Summary: LLM-AggreFact test, ENTAILMENT-ONLY (no GED), windowed {WINDOW}/{OVERLAP} words, n={len(test_df)}

| | threshold | acc | bal. acc | precision | recall | F1 | AUROC | AUPRC |
|---|---|---|---|---|---|---|---|---|
| old truncation (full test) | 0.40 | {OLD_BASELINE["accuracy"]:.4f} | {OLD_BASELINE["balanced_accuracy"]:.4f} | {OLD_BASELINE["precision"]:.4f} | {OLD_BASELINE["recall"]:.4f} | {OLD_BASELINE["f1"]:.4f} | n/a | n/a |
| windowed, fixed | 0.40 | {res_fixed["accuracy"]:.4f} | {res_fixed["balanced_accuracy"]:.4f} | {res_fixed["precision"]:.4f} | {res_fixed["recall"]:.4f} | {res_fixed["f1"]:.4f} | {res_fixed["auroc"]:.4f} | {res_fixed["auprc"]:.4f} |
| windowed, DEV-calibrated | {THR:.3f} | {res_cal["accuracy"]:.4f} | {res_cal["balanced_accuracy"]:.4f} | {res_cal["precision"]:.4f} | {res_cal["recall"]:.4f} | {res_cal["f1"]:.4f} | {res_cal["auroc"]:.4f} | {res_cal["auprc"]:.4f} |

Bootstrap 95% CI ({ci_cal["n_boot"]} resamples) at the calibrated threshold: balanced accuracy {ci_cal["balanced_accuracy_95ci"]}, AUROC {ci_cal["auroc_95ci"]}.
Majority-class baseline (always "{majority["predict"]}"): accuracy {majority["accuracy"]:.4f}, balanced accuracy 0.5.

**Honest scope note.** This scores only the NLI cross-encoder (`{MODEL_NAME}`), not the GED structural comparison, so it
is not a result for the full pipeline. The model is DeBERTa-v3-base NLI, not MiniCheck; published MiniCheck numbers are
not directly comparable. The threshold was selected on DEV and applied once to TEST; 0.40 was not tuned here.
Because the claim is wrapped by the repo's `_make_hypothesis` templating (`HYPOTHESIS_MODE="{HYPOTHESIS_MODE}"`), part of the
gap to a plain-NLI setup may come from the template; try `HYPOTHESIS_MODE="raw"` as an ablation.
'''))